In [ ]:
import json
import csv
import time
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ================= 1. 配置区 =================
CONFIGS = {
    #"Kimi": {"api_key": "你的KEY", "base_url": "https://api.moonshot.cn/v1", "model": "moonshot-v1-8k"},
}

# 并发设置：建议 5-10。如果报错 429 (Rate Limit) 请调小。
MAX_WORKERS = 10 

JSON_FILE = "Product Description.json"
OUTPUT_CSV = "Kimi Results.csv"
ITERATIONS = 100 

# ================= 2. 提问模板 (保持不变) =================
GUIDE_QUERIES = {
    "Apparel": {
        "1.1": {"CN": "我想买一双日常通勤穿的运动鞋，预算中等价位，脚比较宽，想要轻便透气、走路不累的款式，有什么推荐？", "EN": "I want to buy daily sneakers, medium price, wide feet, breathable and comfortable. Any recommendations?"},
        "1.2": {"CN": "我想在 Adidas 的跑鞋里选一双，预算在 700-800 元左右，主要用于全马和半马的日常备赛训练。我比较重视推进感和抓地力，能不能帮我筛选一下并说明理由？", "EN": "I want to choose Adidas running shoes, budget 700-800, for marathon training. I value propulsion and grip. Can you recommend one?"},
        "1.3": {"CN": "我目前想在 Champion Champracer PRO1 和 Nike Air Zoom Upturn SC 之间选一双。我主要用于日常走路通勤，预算比较有限。我非常看重复古颜值和缓震感。针对性价比以及夏天不闷脚，哪个更好？", "EN": "Hesitating between Champion PRO1 and Nike Air Zoom. For commuting, limited budget. Which one is better for cost-performance and breathability?"}
    },
    "Cosmetics": {
        "1.1": {"CN": "我想买一款适合油性/混油皮肤的粉底液，要求持久控油、不容易暗沉，价格中等偏上，有什么品牌可以考虑？", "EN": "Looking for foundation for oily skin, long-lasting oil control, anti-dulling. Any suggestions?"},
        "1.2": {"CN": "我想买一款 兰蔻 的粉底液，预算 500 元以内，我属于典型的混油皮，T区经常出油且容易脱妆。我比较重视持久度和不卡粉，能帮我说明它的持妆技术吗？", "EN": "Want Lancôme foundation under 500 for oily skin. I value longevity and no-cakey. Can you explain its technology?"},
        "1.3": {"CN": "我在纠结买 香奈儿“金砖”粉底液 还是 欧莱雅“小金牌”粉底液。我属于熟龄干性皮肤，看重抗老修护。针对卸妆后是否暗沉以及减少细纹，哪一个更让你信赖？", "EN": "Chanel vs L'Oreal Golden Lift. Mature dry skin, anti-aging. Which one is more trustworthy for anti-dulling and wrinkle reduction?"}
    },
    "Electronics": {
        "1.1": {"CN": "想换一部手机，预算 4000 左右，要求主要是拍照和电池续航要好一点，系统流畅，平时也打游戏。有什么建议？", "EN": "New phone, budget 4000, good photo and battery, smooth OS. Any advice?"},
        "1.2": {"CN": "我想买一款 小米 的高端旗舰手机，预算 5500 元左右。我平时经常自拍，且需要快速查看外卖和打车动态。听说小米有带背屏的手机，能推荐并说明理由吗？", "EN": "Want Xiaomi flagship, 5500 budget. I like selfies and need to check delivery status. Recommend one with a back screen?"},
        "1.3": {"CN": "我目前在 iPhone 17 Pro Max 和 Xiaomi 17 Pro Max 之间犹豫。我主要用于专业摄影和高强度办公，看重长焦和续航。针对出差不充电和演唱会拍摄，哪个更适合我？", "EN": "iPhone 17 PM vs Xiaomi 17 PM. Professional photo and office use. Which one is better for long trips and concert shooting?"}
    }
}

# 线程锁：保证 CSV 写入安全
csv_lock = threading.Lock()

def fetch_ai_response(task):
    """
    单个请求任务函数
    """
    model_name, cfg, cat_key, lang_key, fmt_key, g_type, run_id, full_prompt = task
    client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])
    
    try:
        res = client.chat.completions.create(
            model=cfg['model'],
            messages=[{"role": "user", "content": full_prompt}],
            temperature=0.9,
            timeout=60 # 设置 60 秒超时
        )
        content = res.choices[0].message.content
        
        # 写入 CSV (加锁)
        with csv_lock:
            with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([model_name, cat_key, lang_key, fmt_key, g_type, run_id, content])
        return True
    except Exception as e:
        print(f"❌ 请求失败 [{model_name} | {cat_key} | {run_id}]: {e}")
        return False

def run_experiment():
    # 1. 加载数据
    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        data_store = json.load(f)

    # 2. 初始化 CSV 表头
    if not os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Model', 'Category', 'Language', 'Format', 'Guide_Type', 'Run_ID', 'Response'])

    # 3. 预构建所有任务
    all_tasks = []
    print("正在构建任务列表...")
    for model_name, cfg in CONFIGS.items():
        for cat_key, prices in data_store.items():
            for lang_key in ["CN", "EN"]:
                for fmt_key in ["FAQ", "List", "Paragraph"]:
                    
                    combined_content = ""
                    for p_level in ["budget", "medium", "premium"]:
                        content = prices.get(p_level, {}).get(lang_key, {}).get(fmt_key, "")
                        if content:
                            combined_content += f"\n[Product Option: {p_level}]\n{content}\n"
                    
                    if not combined_content: continue

                    for g_type in ["1.1", "1.2", "1.3"]:
                        query = GUIDE_QUERIES[cat_key][g_type][lang_key]
                        full_prompt = f"Available Products:\n{combined_content}\n\nUser Question: {query}"
                        
                        for i in range(1, ITERATIONS + 1):
                            all_tasks.append((model_name, cfg, cat_key, lang_key, fmt_key, g_type, i, full_prompt))

    total_tasks = len(all_tasks)
    print(f"🚀 任务构建完成，共计 {total_tasks} 条请求。开始并发采集...")

    # 4. 使用线程池执行任务
    completed_count = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 提交所有任务
        future_to_task = {executor.submit(fetch_ai_response, task): task for task in all_tasks}
        
        # 监控进度
        for future in as_completed(future_to_task):
            completed_count += 1
            if completed_count % 10 == 0:
                print(f"📊 进度: {completed_count}/{total_tasks} ({completed_count/total_tasks:.2%})")

if __name__ == "__main__":
    run_experiment()